In [24]:
!pip -q install fastapi uvicorn pydantic==2.* transformers torch pyngrok requests

In [25]:
import pathlib

ROOT = pathlib.Path("trigger-warning-app")
BACKEND = ROOT / "backend"
APP = BACKEND / "app"
PIPE = APP / "pipeline"

PIPE.mkdir(parents=True, exist_ok=True)
(APP / "__init__.py").write_text("")
(PIPE / "__init__.py").write_text("")

# ---------- schemas.py ----------
(PIPE / "schemas.py").write_text(r'''
from pydantic import BaseModel, Field
from typing import List, Dict, Optional

class ModerateRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=20000)
    locale: Optional[str] = "en"

class Finding(BaseModel):
    category: str
    severity: float  # 0..1
    reasons: List[str] = []

class ModerateResponse(BaseModel):
    safe_to_show: bool
    overall_severity: float
    findings: List[Finding]
    trigger_warnings: List[str]
    debug: Optional[Dict] = None
''')

# ---------- rules.py (no slur lists in public repo) ----------
(PIPE / "rules.py").write_text(r'''
import re
from typing import List, Tuple, Dict

SCAM_PATTERNS: List[Tuple[str, re.Pattern]] = [
    ("payment_gift_cards", re.compile(r"\b(gift\s*card|steam\s*card|itunes\s*card)\b", re.I)),
    ("urgent_act_now", re.compile(r"\b(act\s+now|urgent|limited\s+time|final\s+notice)\b", re.I)),
    ("crypto_payment", re.compile(r"\b(bitcoin|btc|usdt|wallet\s+address)\b", re.I)),
    ("impersonation", re.compile(r"\b(verify\s+your\s+account|account\s+will\s+be\s+closed)\b", re.I)),
    ("too_good_to_be_true", re.compile(r"\b(guaranteed\s+profit|double\s+your\s+money|100%\s+returns)\b", re.I)),
]

SEXUAL_HINT_PATTERNS: List[Tuple[str, re.Pattern]] = [
    ("explicit_solicitation", re.compile(r"\b(send\s+nudes|hook\s*up\s+now|dm\s+for\s+sex)\b", re.I)),
]

CHILD_SAFETY_PATTERNS: List[Tuple[str, re.Pattern]] = [
    ("minor_age_mentions", re.compile(r"\b(i'?m\s+)?(1[0-7]|under\s*18)\b", re.I)),
    ("adult_minor_context", re.compile(r"\b(older\s+men|older\s+women|age\s+gap)\b", re.I)),
]

def run_rule_checks(text: str) -> Dict[str, list]:
    hits = {"scam": [], "sexual": [], "child_safety": []}
    for name, pat in SCAM_PATTERNS:
        if pat.search(text):
            hits["scam"].append(name)
    for name, pat in SEXUAL_HINT_PATTERNS:
        if pat.search(text):
            hits["sexual"].append(name)
    for name, pat in CHILD_SAFETY_PATTERNS:
        if pat.search(text):
            hits["child_safety"].append(name)
    return hits
''')

# ---------- heuristics.py ----------
(PIPE / "heuristics.py").write_text(r'''
from typing import List

def scam_heuristic_score(hits: List[str]) -> float:
    score = 0.0
    score += 0.35 if "payment_gift_cards" in hits else 0.0
    score += 0.25 if "crypto_payment" in hits else 0.0
    score += 0.20 if "urgent_act_now" in hits else 0.0
    score += 0.20 if "impersonation" in hits else 0.0
    score += 0.15 if "too_good_to_be_true" in hits else 0.0
    return min(score, 1.0)

def child_safety_score(hits: List[str]) -> float:
    score = 0.0
    score += 0.60 if "minor_age_mentions" in hits else 0.0
    score += 0.20 if "adult_minor_context" in hits else 0.0
    return min(score, 1.0)

def sexual_score(hits: List[str]) -> float:
    return 0.55 if "explicit_solicitation" in hits else 0.0
''')

# ---------- models.py (LOCAL AI via transformers) ----------
(PIPE / "models.py").write_text(r'''
from typing import Dict
from transformers import pipeline

# Loads once (slow the first time, fast after)
# This model is trained for toxicity/harassment signals.
toxicity_model = pipeline(
    "text-classification",
    model="unitary/toxic-bert",
    truncation=True
)

def model_moderation(text: str, locale: str = "en") -> Dict[str, float]:
    """
    Local moderation using Toxic-BERT.
    Returns scores compatible with your pipeline.
    """
    r = toxicity_model(text)[0]
    label = str(r.get("label", "")).lower()
    score = float(r.get("score", 0.0))

    hate = 0.0
    harassment = 0.0

    # Toxic-BERT label names can vary; keep logic robust.
    # We treat toxic as harassment; severe toxic as higher severity.
    if "toxic" in label:
        harassment = max(harassment, score)
    if "severe" in label:
        hate = max(hate, score)

    return {
        "hate": hate,
        "harassment": harassment,
        "sexual": 0.0,
        "child_sexual_content": 0.0,
        "violence": 0.0,
        "self_harm": 0.0,
    }
''')

# ---------- policy.py ----------
(PIPE / "policy.py").write_text(r'''
from typing import List
from app.pipeline.schemas import Finding, ModerateResponse
from app.pipeline.rules import run_rule_checks
from app.pipeline.heuristics import scam_heuristic_score, child_safety_score, sexual_score
from app.pipeline.models import model_moderation

def run_pipeline(text: str, locale: str = "en") -> ModerateResponse:
    rule_hits = run_rule_checks(text)
    model_scores = model_moderation(text, locale)

    findings: List[Finding] = []

    scam_score = scam_heuristic_score(rule_hits["scam"])
    if scam_score > 0:
        findings.append(Finding(category="scam_fraud", severity=scam_score, reasons=rule_hits["scam"]))

    child_score = child_safety_score(rule_hits["child_safety"])
    # If you later add a child safety model, combine it here.
    if child_score > 0:
        findings.append(Finding(category="child_safety", severity=child_score, reasons=rule_hits["child_safety"]))

    sex_score = sexual_score(rule_hits["sexual"])
    if sex_score > 0:
        findings.append(Finding(category="sexual_content", severity=sex_score, reasons=rule_hits["sexual"]))

    hate_score = max(model_scores.get("hate", 0.0), model_scores.get("harassment", 0.0))
    if hate_score > 0:
        findings.append(Finding(category="hate_harassment", severity=hate_score, reasons=["local_model_signal"]))

    overall = 0.0 if not findings else max(f.severity for f in findings)

    trigger_warnings = []
    if any(f.category == "child_safety" and f.severity >= 0.4 for f in findings):
        trigger_warnings.append("Child safety risk — may involve minors, grooming, or exploitation.")
    if any(f.category == "sexual_content" and f.severity >= 0.4 for f in findings):
        trigger_warnings.append("Sexual content — may be explicit, suggestive, or solicitation.")
    if any(f.category == "hate_harassment" and f.severity >= 0.4 for f in findings):
        trigger_warnings.append("Hate/harassment — may include hateful or abusive language.")
    if any(f.category == "scam_fraud" and f.severity >= 0.4 for f in findings):
        trigger_warnings.append("Possible scam/fraud — be cautious with links, payments, and personal info.")

    safe_to_show = not (overall >= 0.7 or any(f.category == "child_safety" and f.severity >= 0.6 for f in findings))

    return ModerateResponse(
        safe_to_show=safe_to_show,
        overall_severity=overall,
        findings=findings,
        trigger_warnings=trigger_warnings,
        debug={"rule_hits": rule_hits, "model_scores": model_scores},
    )
''')

# ---------- main.py ----------
(APP / "main.py").write_text(r'''
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from app.pipeline.schemas import ModerateRequest, ModerateResponse
from app.pipeline.policy import run_pipeline

app = FastAPI(title="Trigger Warning Moderation API", version="0.1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/moderate", response_model=ModerateResponse)
def moderate(req: ModerateRequest):
    return run_pipeline(req.text, req.locale)
''')

print("✅ Backend created:", ROOT)

✅ Backend created: trigger-warning-app


In [26]:
import sys, pathlib
sys.path.append(str(pathlib.Path("trigger-warning-app/backend")))

from app.pipeline.policy import run_pipeline

tests = [
    "URGENT verify your account pay with gift card",
    "send nudes now",
    "i'm 16 and older men messaging me",
    "I hate you and you are disgusting",
    "normal coffee post"
]
for t in tests:
    r = run_pipeline(t, "en")
    print("\nTEXT:", t)
    print("WARNINGS:", r.trigger_warnings)
    print("FINDINGS:", [(f.category, round(f.severity,2), f.reasons) for f in r.findings])


TEXT: URGENT verify your account pay with gift card
WARNINGS: ['Possible scam/fraud — be cautious with links, payments, and personal info.']
FINDINGS: [('scam_fraud', 0.75, ['payment_gift_cards', 'urgent_act_now', 'impersonation'])]

TEXT: send nudes now
WARNINGS: ['Sexual content — may be explicit, suggestive, or solicitation.']
FINDINGS: [('sexual_content', 0.6, ['explicit_solicitation'])]

TEXT: i'm 16 and older men messaging me
WARNINGS: ['Child safety risk — may involve minors, grooming, or exploitation.']
FINDINGS: [('child_safety', 0.7, ['minor_age_mentions'])]

TEXT: I hate you and you are disgusting
WARNINGS: []
FINDINGS: []

TEXT: normal coffee post
WARNINGS: []
FINDINGS: []


In [ ]:
!uvicorn app.main:app --app-dir trigger-warning-app/backend --host 0.0.0.0 --port 8000

config.json: 100% 811/811 [00:00<00:00, 3.28MB/s]
model.safetensors: 100% 438M/438M [00:09<00:00, 46.1MB/s]
Loading weights: 100% 201/201 [00:00<00:00, 878.66it/s, Materializing param=classifier.weight]
BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100% 174/174 [00:00<00:00, 568kB/s]
vocab.txt: 232kB [00:00, 9.52MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 380kB/s]
INFO:     Started server process [11606]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("PASTE_YOUR_REAL_NGROK_TOKEN_HERE")
public_url = ngrok.connect(8000, "http")
print("🌍 PUBLIC URL:", public_url)

In [ ]:
import requests
base = str(public_url).rstrip("/")
print(requests.get(base + "/health").json())
print(requests.post(base + "/moderate", json={"text":"send nudes now","locale":"en"}).json())

In [ ]:
from pathlib import Path
import zipfile

EXT = Path("instagram-trigger-extension")
EXT.mkdir(exist_ok=True)

(EXT / "manifest.json").write_text(r'''
{
  "manifest_version": 3,
  "name": "Instagram Trigger Warning Scanner",
  "version": "0.1.0",
  "description": "Scans visible Instagram captions and displays trigger warnings.",
  "host_permissions": [
    "https://www.instagram.com/*",
    "https://*.ngrok-free.app/*"
  ],
  "content_scripts": [
    {
      "matches": ["https://www.instagram.com/*"],
      "js": ["content.js"],
      "css": ["styles.css"],
      "run_at": "document_idle"
    }
  ]
}
''')

API_BASE = str(public_url).rstrip("/")

(EXT / "content.js").write_text(f'''
const API_BASE = "{API_BASE}";
const RESCAN_MS = 2500;
const cache = new Map();

function cleanText(t) {{ return (t||"").replace(/\\s+/g," ").trim(); }}

async function moderateText(text) {{
  const now = Date.now();
  const cached = cache.get(text);
  if (cached && (now - cached.ts) < 60000) return cached.result;

  const res = await fetch(`${{API_BASE}}/moderate`, {{
    method: "POST",
    headers: {{ "Content-Type":"application/json" }},
    body: JSON.stringify({{ text, locale:"en" }})
  }});
  if (!res.ok) throw new Error("API error " + res.status);
  const data = await res.json();
  cache.set(text, {{ ts: now, result: data }});
  return data;
}}

function makeBanner(warnings, overall) {{
  const div = document.createElement("div");
  div.className = "tw-banner";
  const title = document.createElement("div");
  title.className = "tw-title";
  title.textContent = `Trigger warning (severity ${{Number(overall).toFixed(2)}})`;
  const ul = document.createElement("ul");
  ul.className = "tw-list";
  warnings.forEach(w => {{ const li=document.createElement("li"); li.textContent=w; ul.appendChild(li); }});
  const btn = document.createElement("button");
  btn.className = "tw-btn";
  btn.textContent = "Hide/Show";
  btn.onclick = () => {{
    const t = div.parentElement?.querySelector(".tw-hide-target");
    if (t) t.classList.toggle("tw-hidden");
  }};
  div.appendChild(title);
  div.appendChild(ul);
  div.appendChild(btn);
  return div;
}}

function getCaptionTextFromArticle(article) {{
  const spans = article.querySelectorAll("span");
  let best = "";
  for (const sp of spans) {{
    const t = cleanText(sp.textContent);
    if (t.length >= 20 && t.length > best.length) best = t;
  }}
  return best;
}}

function wrapHideTarget(article) {{
  if (article.querySelector(".tw-hide-target")) return;
  const w = document.createElement("div");
  w.className = "tw-hide-target";
  Array.from(article.childNodes).forEach(n => w.appendChild(n));
  article.appendChild(w);
}}

async function scan() {{
  const articles = document.querySelectorAll("article");
  for (const article of articles) {{
    if (article.dataset.twProcessed === "1") continue;
    const caption = getCaptionTextFromArticle(article);
    if (!caption) continue;

    try {{
      const result = await moderateText(caption);
      if (result.trigger_warnings?.length) {{
        wrapHideTarget(article);
        article.prepend(makeBanner(result.trigger_warnings, result.overall_severity || 0));
        if (result.safe_to_show === false) {{
          article.querySelector(".tw-hide-target")?.classList.add("tw-hidden");
        }}
      }}
      article.dataset.twProcessed = "1";
    }} catch (e) {{
      console.warn("TW extension error:", e);
    }}
  }}
}}

setInterval(scan, RESCAN_MS);
scan();
''')

(EXT / "styles.css").write_text(r'''
.tw-banner { font-family: system-ui; border:1px solid rgba(255,0,0,.35); border-radius:10px; padding:10px 12px; margin:8px 0; background:rgba(255,0,0,.06); }
.tw-title { font-weight:700; margin-bottom:6px; }
.tw-list { margin:0 0 8px 18px; padding:0; }
.tw-btn { border:1px solid rgba(0,0,0,.2); border-radius:8px; padding:6px 10px; cursor:pointer; background:white; }
.tw-hide-target.tw-hidden { filter: blur(10px); pointer-events:none; user-select:none; }
''')

zip_name = "instagram-trigger-extension.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in EXT.rglob("*"):
        z.write(p, arcname=str(p.relative_to(Path("."))))

print("✅ Extension zip created:", zip_name)
print("✅ API_BASE:", API_BASE)

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path("trigger-warning-app/backend")))

from app.pipeline.policy import run_pipeline

print(run_pipeline("send nudes now").trigger_warnings)
print(run_pipeline("URGENT verify account pay gift card").trigger_warnings)
print(run_pipeline("I hate you").trigger_warnings)

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("PASTE_YOUR_REAL_NGROK_TOKEN")

public_url = ngrok.connect(8000, "http")
print("🌍 PUBLIC URL:", public_url)

In [ ]:
import requests

base = str(public_url).rstrip("/")

print(requests.get(base + "/health").json())
print(requests.post(
    base + "/moderate",
    json={"text": "send nudes now", "locale": "en"}
).json())